<div style="align: center;">
    <br>
    <img src="https://storage.googleapis.com/kaggle-datasets-images/2289007/3846912/ad5e128929f5ac26133b67a6110de7c0/dataset-cover.jpg?" style="display:block; margin:auto; width:75%; height:350px;">
</div><br><br> 

<div style="letter-spacing:normal; opacity:1.;">
<!--   https://xkcd.com/color/rgb/   -->
  <p style="text-align:center; background-color: lightsalmon; color: Jaguar; border-radius:10px; font-family:monospace; 
            line-height:1.4; font-size:32px; font-weight:bold; text-transform: uppercase; padding: 9px;">
            <strong>Finance Company Credit-Related Information</strong></p>  
  
  <p style="text-align:center; background-color:romance; color: Jaguar; border-radius:10px; font-family:monospace; 
            line-height:1.0; font-size:28px; font-weight:normal; text-transform: capitalize; padding: 5px;"
     >Machine Learning Module: Credit Score Classification Part 1: Data Cleaning<br>Data Cleaning & Analysis with Python</p>    
</div>

https://celik-muhammed.medium.com/how-to-converting-pandas-column-of-comma-separated-strings-into-dummy-variables-762c02282a6c

# 01. Importing Related Libraries

- Once you've Installed NumPy and Pandas etc. you can Import them as a Library.
- Reading the Data from File

In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

# import warnings
# # Suppressing a warning 
# warnings.filterwarnings("ignore") 
# warnings.warn("this will not show")

import re
import time
import random
import tempfile
from tqdm.notebook import tqdm

import gc
gc.collect()

## Reading the Data from File

In [ ]:
df_origin_train = pd.read_csv('../input/credit-score-classification/train.csv')
df_train = df_origin_train.copy()
df_train

In [ ]:
df_origin_test = pd.read_csv('../input/credit-score-classification/test.csv')
df_test = df_origin_test.copy()
df_test

# A. Recognizing and Understanding Data

## Check the head, shape, data-types of the features.

In [ ]:
df_train.shape, df_test.shape

In [ ]:
display(
    df_train.info(), 
    print(), 
    df_test.info()
)

## Check the statistical values of features.

In [ ]:
display(
    df_train.describe().T, 
    print(), 
    df_test.describe().T
)

In [ ]:
display(
    df_train.describe(exclude=np.number).T, 
    print(), 
    df_test.describe(exclude=np.number).T
)

## Basically check the missing values

In [ ]:
df_train['Credit_Score'].isna().sum()

## Check Column Names Concat Train-Test Data

In [ ]:
(df_train.columns[:-1]!=df_test.columns).sum()

In [ ]:
df = pd.concat([df_train, df_test], ignore_index=True)
df.shape

In [ ]:
df['Credit_Score'].isna().sum()

In [ ]:
df.isna().sum()

In [ ]:
df.isnull().mean()*100

## If needed, rename the columns' names for easy use.

In [ ]:
df.columns

# B. Examining the Data

In [ ]:
df.select_dtypes('O').info()

## Detect strange values apart from the NaN Values

In [ ]:
object_col = df.describe(include='O').columns
object_col

In [ ]:
for col in object_col:
    print('Column Name: '+col)
    print("**"*20)
    print(df_train[col].value_counts(dropna=False))
    print('END', "--"*18, '\n')

In [ ]:
df_copy1 = df.copy()
df_copy1.shape

## Clear strange values apart from the NaN Values

In [ ]:
def text_cleaning(data):
    if data is np.NaN or not isinstance(data, str):
        return data
    else:
        return str(data).strip('_ ,"')

In [ ]:
df = df_copy1.applymap(text_cleaning).replace(['', 'nan', '!@9#%8', '#F%$D@*&8'], np.NaN)
df

In [ ]:
df.isna().sum()

# C. Fixing a data type (value_counts(), astype(), infer_objects(), convert_dtypes())

- Detect strange values by Columns Convert Object Types to Numeric Types (int, float, np.int64, pd.Int64Dtype())
- Combining object columns

Change column type in pandas:

  01. astype() - convert (almost) any type to (almost) any other type (even if it's not necessarily sensible to do so). Also   allows you to convert to categorial types (very useful).
  02. infer_objects() - a utility method to convert object columns holding Python objects to a pandas type if possible.
  03. convert_dtypes() - convert DataFrame columns to the "best possible" dtype that supports pd.NA (pandas' object to indicate a missing value).
  04. to_numeric() - provides functionality to safely convert non-numeric types.(See also to_datetime() and to_timedelta().)
  05. factorize() - provides sorting

In [ ]:
df.select_dtypes('O').info()

In [ ]:
df['ID']                      = df.ID.apply(lambda x: int(x, 16))
df['Customer_ID']             = df.Customer_ID.apply(lambda x: int(x[4:], 16))
df['Month']                   = pd.to_datetime(df.Month, format='%B').dt.month
df['Age']                     = df.Age.astype(int) 
df['SSN']                     = df.SSN.apply(lambda x: x if x is np.NaN else int(str(x).replace('-', ''))).astype(float)
df['Annual_Income']           = df.Annual_Income.astype(float)
df['Num_of_Loan']             = df.Num_of_Loan.astype(int) 
df['Num_of_Delayed_Payment']  = df.Num_of_Delayed_Payment.astype(float)
df['Changed_Credit_Limit']    = df.Changed_Credit_Limit.astype(float)
df['Outstanding_Debt']        = df.Outstanding_Debt.astype(float)
df['Amount_invested_monthly'] = df.Amount_invested_monthly.astype(float)
df['Monthly_Balance']         = df.Monthly_Balance.astype(float)

### Assign Categorical Types to Numeric Types
- Maybe Use Before Machine Learning or Use OrdinalEncoder, LabelEncoder etc.

In [ ]:
# df['Occupation_Num'] = df.Occupation.astype('category').cat.codes
# df['Credit_Mix_Num'] = df.Credit_Mix.astype('category').cat.codes
# df['Payment_of_Min_Amount_Num'] = df.Payment_of_Min_Amount.astype('category').cat.codes
# df['Payment_Behaviour_Num'] = df.Payment_Behaviour.astype('category').cat.codes

### Combining object columns

#### Credit_History_Age

In [ ]:
def Month_Converter(x):
    if pd.notnull(x):
        num1 = int(x.split(' ')[0])
        num2 = int(x.split(' ')[3])
      
        return (num1*12)+num2
    else:
        return x
    
# Month_Converter('3 Years and 1 Months')

In [ ]:
df['Credit_History_Age'] = df.Credit_History_Age.apply(lambda x: Month_Converter(x)).astype(float)

In [ ]:
df.groupby('Customer_ID')['Credit_History_Age'].apply(list)

#### Type_of_Loan

- https://celik-muhammed.medium.com/how-to-converting-pandas-column-of-comma-separated-strings-into-dummy-variables-762c02282a6c

In [ ]:
df['Type_of_Loan'].value_counts(dropna=False).head(20)

In [ ]:
df['Type_of_Loan'] = df['Type_of_Loan'].apply(lambda x: x.lower().replace('and ', '').replace(', ', ',').strip() if pd.notna(x) else x)

In [ ]:
df.groupby('Customer_ID')['Type_of_Loan'].value_counts(dropna=False)

In [ ]:
df.groupby('Customer_ID')['Type_of_Loan'].apply(list)

In [ ]:
def get_Diff_Values_Colum(df_column, diff_value=[], sep=',', replace=''):   
    column = df_column.dropna()
    for i in column:
        if sep not in i and i not in diff_value:
            diff_value.append(i)
        else:
            for data in map(lambda x:x.strip(), re.sub(replace, '', i).split(sep)):
                if not data in diff_value:
                    diff_value.append(data)
    return dict(enumerate(sorted(diff_value)))

In [ ]:
get_Diff_Values_Colum(df['Type_of_Loan'])

# 02. Exploratory Data Analysis (EDA)
- Detect NaN Values and Fill by Customer_ID Group

# A. Object Column NaN Values: Reassign Group Mode Values

In [ ]:
# Reassign and Show Function
def Object_NaN_Values_Reassign_Group_Mode(df, groupby, column, inplace=True):      
    import scipy.stats as stats
    # Assigning Wrong values Make Simple Function
    def make_NaN_and_fill_mode(df, groupby, column, inplace=True):
        # Assign None to np.NaN
        if df[column].isin([None]).sum():
            df[column][df[column].isin([None])] = np.NaN
            
        # fill with local mode
        result = df.groupby(groupby)[column].transform(lambda x: x.fillna(stats.mode(x)[0][0]))

        # inplace
        if inplace:
            df[column]=result
        else:
            return result
    
    # Run      
    if inplace:  
        # Before Assigning NaN values   
        if df[column].value_counts(dropna=False).index.isna().sum():
            x = df[column].value_counts(dropna=False).loc[[np.NaN]]
            print(f'\nBefore Assigning: {column}:', f'have {x.values[0]} NaN Values', end='\n')
            
        a = df.groupby(groupby)[column].apply(list) 
        print(f'\nBefore Assigning Example {column}:\n', *a.head().values, sep='\n', end='\n')
        
        # Assigning
        make_NaN_and_fill_mode(df, groupby, column, inplace)
        
        # After Assigning NaN values
        if df[column].value_counts(dropna=False).index.isna().sum():
            y = df[column].value_counts(dropna=False).loc[[np.NaN]]
            print(f'\nBefore Assigning: {column}:', f'have {y.values[0]} NaN Values', end='\n')
            
        b = df.groupby(groupby)[column].apply(list)
        print(f'\nAfter Assigning Example {column}:\n', *b.head().values, sep='\n', end='\n')
    else:   
        # Show
        return make_NaN_and_fill_mode(df, groupby, column, inplace)

In [ ]:
df_copy2 = df.copy()
df_copy2.shape

In [ ]:
df = df_copy2
df.info()

### Name

In [ ]:
df['Name'].value_counts(dropna=False).head()

In [ ]:
Object_NaN_Values_Reassign_Group_Mode(df, 'Customer_ID', 'Name')

### Occupation

In [ ]:
df['Occupation'].value_counts(dropna=False)

In [ ]:
Object_NaN_Values_Reassign_Group_Mode(df, 'Customer_ID', 'Occupation')

### Type_of_Loan

In [ ]:
df.groupby('Customer_ID')['Type_of_Loan'].value_counts(dropna=False)

In [ ]:
df['Type_of_Loan'].replace([np.NaN], 'No Data', inplace=True)

### Credit_Mix

In [ ]:
df['Credit_Mix'].value_counts(dropna=False)

In [ ]:
Object_NaN_Values_Reassign_Group_Mode(df, 'Customer_ID', 'Credit_Mix')

### Payment_of_Min_Amount

In [ ]:
# Not Required
df['Payment_of_Min_Amount'].value_counts(dropna=False)

### Payment_Behaviour

In [ ]:
df['Payment_Behaviour'].value_counts(dropna=False)

In [ ]:
Object_NaN_Values_Reassign_Group_Mode(df, 'Customer_ID', 'Payment_Behaviour')

# B. Numeric Column NaN Values: Reassign Group Min-Max

In [ ]:
df_copy3 = df.copy()
df_copy3.shape

In [ ]:
df = df_copy3
df.info()

In [ ]:
df['Customer_ID'].nunique()

In [ ]:
# Define Outlier Range
def get_iqr_lower_upper(df, column, multiply=1.5):
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 -q1
    
    lower = q1-iqr*multiply
    upper = q3+iqr*multiply
    affect = df.loc[(df[column]<lower)|(df[column]>upper)].shape
    print('Outliers:', affect)
    return lower, upper

In [ ]:
# Reassign Wrong Values and Show Function
def Numeric_Wrong_Values_Reassign_Group_Min_Max(df, groupby, column, inplace=True):      
    import scipy.stats as stats 

    # Identify Wrong values Range
    def get_group_min_max(df, groupby, column):            
        cur = df[df[column].notna()].groupby(groupby)[column].apply(list)
        x, y = cur.apply(lambda x: stats.mode(x)).apply([min, max])
        return x[0][0], y[0][0]
    
    # Assigning Wrong values
    def make_group_NaN_and_fill_mode(df, groupby, column, inplace=True):
        df_dropped = df[df[column].notna()].groupby(groupby)[column].apply(list)
        x, y = df_dropped.apply(lambda x: stats.mode(x)).apply([min, max])
        mini, maxi = x[0][0], y[0][0]

        # assign Wrong Values to NaN
        col = df[column].apply(lambda x: np.NaN if ((x<mini)|(x>maxi)) else x)

        # fill with local mode
        mode_by_group = df.groupby(groupby)[column].transform(lambda x: x.mode()[0] if not x.mode().empty else np.NaN)
        result = col.fillna(mode_by_group)

        # inplace
        if inplace:
            df[column]=result
        else:
            return result
        
    
    # Run      
    if inplace:   
        # Before Assigning NaN values   
        if df[column].value_counts(dropna=False).index.isna().sum():
            x = df[column].value_counts(dropna=False).loc[[np.NaN]]
            print(f'\nBefore Assigning: {column}:', f'have {x.values[0]} NaN Values', end='\n')
            
        print("\nExisting Min, Max Values:", df[column].apply([min, max]), sep='\n', end='\n')       
        mini, maxi = get_group_min_max(df, groupby, column)        
        print(f"\nGroupby by {groupby}'s Actual min, max Values:", f'min:\t{mini},\nmax:\t{ maxi}', sep='\n', end='\n')        
        
        a = df.groupby(groupby)[column].apply(list) 
        print(f'\nBefore Assigning Example {column}:\n', *a.head().values, sep='\n', end='\n')
        
        # Assigning
        make_group_NaN_and_fill_mode(df, groupby, column, inplace)
        
        # After Assigning NaN values
        if df[column].value_counts(dropna=False).index.isna().sum():
            y = df[column].value_counts(dropna=False).loc[[np.NaN]]
            print(f'\nBefore Assigning: {column}:', f'have {y.values[0]} NaN Values', end='\n')
        
        b = df.groupby(groupby)[column].apply(list)
        print(f'\nAfter Assigning Example {column}:\n', *b.head().values, sep='\n', end='\n')
    else:   
        # Show
        return make_group_NaN_and_fill_mode(df, groupby, column, inplace)

In [ ]:
df.describe().columns

### ID

In [ ]:
df['ID'].nunique()

### Month

In [ ]:
df['Month'].value_counts()

### Age

In [ ]:
Numeric_Wrong_Values_Reassign_Group_Min_Max(df, 'Customer_ID', 'Age')

In [ ]:
# Check Outlier
get_iqr_lower_upper(df, 'Age')

### SSN

In [ ]:
df.SSN.value_counts(dropna=False)

In [ ]:
Numeric_Wrong_Values_Reassign_Group_Min_Max(df, 'Customer_ID', 'SSN')

### Annual_Income

In [ ]:
df.Annual_Income.value_counts(dropna=False)

In [ ]:
Numeric_Wrong_Values_Reassign_Group_Min_Max(df, 'Customer_ID', 'Annual_Income')

### Monthly_Inhand_Salary

In [ ]:
df.Monthly_Inhand_Salary.value_counts(dropna=False)

In [ ]:
Numeric_Wrong_Values_Reassign_Group_Min_Max(df, 'Customer_ID', 'Monthly_Inhand_Salary')

### Num_Bank_Accounts

In [ ]:
df.Num_Bank_Accounts.value_counts(dropna=False)

In [ ]:
Numeric_Wrong_Values_Reassign_Group_Min_Max(df, 'Customer_ID', 'Num_Bank_Accounts')

### Num_Credit_Card

In [ ]:
df_train.Num_Credit_Card.value_counts(dropna=False)

In [ ]:
Numeric_Wrong_Values_Reassign_Group_Min_Max(df, 'Customer_ID', 'Num_Credit_Card')

### Interest_Rate

In [ ]:
df.Interest_Rate.value_counts(dropna=False)

In [ ]:
Numeric_Wrong_Values_Reassign_Group_Min_Max(df, 'Customer_ID', 'Interest_Rate')

### Num_of_Loan

In [ ]:
df.Num_of_Loan.value_counts(dropna=False)

In [ ]:
Numeric_Wrong_Values_Reassign_Group_Min_Max(df, 'Customer_ID', 'Num_of_Loan')

### Delay_from_due_date

In [ ]:
df.Delay_from_due_date.value_counts(dropna=False)

In [ ]:
Numeric_Wrong_Values_Reassign_Group_Min_Max(df, 'Customer_ID', 'Delay_from_due_date')

### Num_of_Delayed_Payment

In [ ]:
df.Num_of_Delayed_Payment.value_counts(dropna=False)

In [ ]:
Numeric_Wrong_Values_Reassign_Group_Min_Max(df, 'Customer_ID', 'Num_of_Delayed_Payment')

### Changed_Credit_Limit

In [ ]:
df.Changed_Credit_Limit.value_counts(dropna=False)

In [ ]:
Numeric_Wrong_Values_Reassign_Group_Min_Max(df, 'Customer_ID', 'Changed_Credit_Limit')

### Num_Credit_Inquiries

In [ ]:
df.Num_Credit_Inquiries.value_counts(dropna=False)

In [ ]:
Numeric_Wrong_Values_Reassign_Group_Min_Max(df, 'Customer_ID', 'Num_Credit_Inquiries')

### Outstanding_Debt

In [ ]:
df.Outstanding_Debt.value_counts(dropna=False)

In [ ]:
Numeric_Wrong_Values_Reassign_Group_Min_Max(df, 'Customer_ID', 'Outstanding_Debt')

### Credit_Utilization_Ratio

In [ ]:
df.Credit_Utilization_Ratio.value_counts(dropna=False)

In [ ]:
df.Credit_Utilization_Ratio.isna().sum()

### Credit_History_Age

In [ ]:
df.Credit_History_Age.value_counts(dropna=False)

In [ ]:
df['Credit_History_Age'] = df.groupby('Customer_ID')['Credit_History_Age'].apply(lambda x: x.interpolate().bfill().ffill())

### Total_EMI_per_month

In [ ]:
df.Total_EMI_per_month.value_counts(dropna=False)

In [ ]:
Numeric_Wrong_Values_Reassign_Group_Min_Max(df, 'Customer_ID', 'Total_EMI_per_month')

### Amount_invested_monthly

In [ ]:
df.Amount_invested_monthly.value_counts(dropna=False)

In [ ]:
Numeric_Wrong_Values_Reassign_Group_Min_Max(df, 'Customer_ID', 'Amount_invested_monthly')

### Monthly_Balance

In [ ]:
df.Monthly_Balance.value_counts(dropna=False)

In [ ]:
Numeric_Wrong_Values_Reassign_Group_Min_Max(df, 'Customer_ID', 'Monthly_Balance')

# C. End of Cleaning

- We fill NaN values from group mode for Object and Numeric features.
- We change group  min max that is outlier or wrong values from numeric features.

- Check before Modeling:
    - Num_Bank_Accounts, Delay_from_due_date, Num_of_Delayed_Payment, Monthly_Balance have negative strange number

In [ ]:
df

In [ ]:
df.isna().sum()

In [ ]:
df.info()

In [ ]:
df.to_csv("clean_credit_score_classification.csv", index=False)

In [ ]:
df = pd.read_csv('clean_credit_score_classification.csv')

# 03. Prepare Modeling
- Num_Bank_Accounts, 
- Delay_from_due_date, 
- Num_of_Delayed_Payment, 
- Monthly_Balance have negative strange number
- Amount_invested_monthly have positive strange number

In [ ]:
df[df['Num_Bank_Accounts']<0]

In [ ]:
df[df['Num_Bank_Accounts']<0]['Customer_ID'].unique()

In [ ]:
df[df['Customer_ID']==22931]

In [ ]:
df.loc[df['Num_Bank_Accounts']<0, 'Num_Bank_Accounts'] = 0

In [ ]:
df[df['Delay_from_due_date']<0]

In [ ]:
df[df['Delay_from_due_date']<0]['Customer_ID'].unique()

In [ ]:
df[df['Customer_ID']==48234].iloc[:,0:15]

In [ ]:
df.loc[df['Delay_from_due_date']<0, 'Delay_from_due_date'] = None

In [ ]:
Numeric_Wrong_Values_Reassign_Group_Min_Max(df, 'Customer_ID', 'Delay_from_due_date')

In [ ]:
df[df['Num_of_Delayed_Payment']<0]

In [ ]:
df[df['Num_of_Delayed_Payment']<0]['Customer_ID'].unique()

In [ ]:
df[df['Customer_ID']==8625].iloc[:,0:20]

In [ ]:
df.loc[df['Num_of_Delayed_Payment']<0, 'Num_of_Delayed_Payment'] = None

In [ ]:
Numeric_Wrong_Values_Reassign_Group_Min_Max(df, 'Customer_ID', 'Num_of_Delayed_Payment')

In [ ]:
df[df['Monthly_Balance']<0]

In [ ]:
df[df['Monthly_Balance']<0]['Customer_ID'].unique()

In [ ]:
df[df['Customer_ID']==23184]

In [ ]:
df.loc[df['Monthly_Balance']<0, 'Monthly_Balance'] = None

In [ ]:
Numeric_Wrong_Values_Reassign_Group_Min_Max(df, 'Customer_ID', 'Monthly_Balance')

In [ ]:
df[df['Amount_invested_monthly']>=10000]

In [ ]:
df[df['Amount_invested_monthly']>=10000]['Customer_ID'].unique()

In [ ]:
df[df['Customer_ID']==44897]

In [ ]:
df['Amount_invested_monthly'].plot(kind='box', vert=0)

In [ ]:
df.loc[df['Amount_invested_monthly']>=10000, 'Amount_invested_monthly'] = None

In [ ]:
df['Amount_invested_monthly'].plot(kind='box', vert=0)

In [ ]:
# fill group Mode
df['Amount_invested_monthly'] = df.groupby('Customer_ID')['Amount_invested_monthly'].transform(lambda x: x.mode()[0] if not x.mode().empty else np.NaN)

In [ ]:
# train check
df[df['Credit_Score'].notna()].info()

In [ ]:
# train save
df[df['Credit_Score'].notna()].to_csv("train.csv", index=False)

In [ ]:
# test check
df[df['Credit_Score'].isna()].info()

In [ ]:
# test save
df[df['Credit_Score'].isna()].drop(columns='Credit_Score').to_csv("test.csv", index=False)

## Download Link

In [ ]:
from IPython.display import FileLink, FileLinks
train_file = FileLink(r'train.csv', result_html_prefix="Click here to download: ")
test_file = FileLink(r'test.csv', result_html_prefix="Click here to download: ")

display(train_file, test_file)

# End of the Project